In [1]:
# ============================================================
# MARCH MACHINE LEARNING MANIA 2026 — FULL PIPELINE (1 CELL)
# ============================================================
import numpy as np
import pandas as pd
import warnings, re
warnings.filterwarnings('ignore')

import lightgbm as lgb
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import brier_score_loss
from sklearn.isotonic import IsotonicRegression
from scipy.special import expit
from scipy.optimize import minimize_scalar

DATA_DIR = "/kaggle/input/competitions/march-machine-learning-mania-2026"
SEED = 42
np.random.seed(SEED)

# ── 1. LOAD DATA ─────────────────────────────────────────────
m_reg_compact   = pd.read_csv(f"{DATA_DIR}/MRegularSeasonCompactResults.csv")
m_reg_detailed  = pd.read_csv(f"{DATA_DIR}/MRegularSeasonDetailedResults.csv")
m_tourney       = pd.read_csv(f"{DATA_DIR}/MNCAATourneyCompactResults.csv")
m_seeds         = pd.read_csv(f"{DATA_DIR}/MNCAATourneySeeds.csv")
m_massey_raw    = pd.read_csv(f"{DATA_DIR}/MMasseyOrdinals.csv")

w_reg_compact   = pd.read_csv(f"{DATA_DIR}/WRegularSeasonCompactResults.csv")
w_reg_detailed  = pd.read_csv(f"{DATA_DIR}/WRegularSeasonDetailedResults.csv")
w_tourney       = pd.read_csv(f"{DATA_DIR}/WNCAATourneyCompactResults.csv")
w_seeds         = pd.read_csv(f"{DATA_DIR}/WNCAATourneySeeds.csv")

sample_sub      = pd.read_csv(f"{DATA_DIR}/SampleSubmissionStage1.csv")
print(f"Loaded. Sub rows: {len(sample_sub)}")

# ── 2. SEED PARSING ──────────────────────────────────────────
def process_seeds(seeds_df):
    df = seeds_df.copy()
    df['SeedNum'] = df['Seed'].apply(lambda s: int(re.search(r'(\d+)', str(s)).group(1)))
    return df[['Season', 'TeamID', 'SeedNum']]

m_seeds_clean = process_seeds(m_seeds)
w_seeds_clean = process_seeds(w_seeds)

# ── 3. TEAM SEASON STATS ─────────────────────────────────────
def compute_team_stats(detailed_df):
    df = detailed_df.copy()
    day_min = df.groupby('Season')['DayNum'].transform('min')
    day_max = df.groupby('Season')['DayNum'].transform('max')
    df['RW'] = 0.5 + (df['DayNum'] - day_min) / (day_max - day_min + 1)

    records = []
    for _, row in df.iterrows():
        w = row['RW']
        for side, opp in [('W', 'L'), ('L', 'W')]:
            pts  = row[f'{side}Score'];  opts = row[f'{opp}Score']
            fga  = row[f'{side}FGA'];   or_  = row[f'{side}OR']
            ftm  = row[f'{side}FTM'];   fta  = row[f'{side}FTA']
            to   = row[f'{side}TO'];    dr   = row[f'{side}DR']
            poss     = fga - or_ + to + 0.475 * fta
            opp_poss = row[f'{opp}FGA'] - row[f'{opp}OR'] + row[f'{opp}TO'] + 0.475 * row[f'{opp}FTA']
            off_eff  = (pts  / poss     * 100) if poss     > 0 else 100.0
            def_eff  = (opts / opp_poss * 100) if opp_poss > 0 else 100.0
            pace     = (poss + opp_poss) / 2
            ts_d     = 2 * (fga + 0.44 * fta)
            ts_pct   = pts / ts_d if ts_d > 0 else 0.5
            tot_r    = or_ + dr;  opp_r = row[f'{opp}OR'] + row[f'{opp}DR']
            reb_pct  = tot_r / (tot_r + opp_r) if (tot_r + opp_r) > 0 else 0.5
            to_pct   = to / poss if poss > 0 else 0.15
            records.append({
                'Season': row['Season'], 'TeamID': row[f'{side}TeamID'],
                'Win': 1 if side == 'W' else 0,
                'OffEff': off_eff, 'DefEff': def_eff, 'NetEff': off_eff - def_eff,
                'Pace': pace, 'TS_pct': ts_pct, 'Reb_pct': reb_pct, 'TO_pct': to_pct, 'W': w
            })

    gdf = pd.DataFrame(records)
    agg = []
    for (season, team), grp in gdf.groupby(['Season', 'TeamID']):
        wts = grp['W'].values
        agg.append({
            'Season': season, 'TeamID': team,
            'WinPct':   np.average(grp['Win'],     weights=wts),
            'OffEff':   np.average(grp['OffEff'],  weights=wts),
            'DefEff':   np.average(grp['DefEff'],  weights=wts),
            'NetEff':   np.average(grp['NetEff'],  weights=wts),
            'Pace':     np.average(grp['Pace'],    weights=wts),
            'TS_pct':   np.average(grp['TS_pct'],  weights=wts),
            'Reb_pct':  np.average(grp['Reb_pct'], weights=wts),
            'TO_pct':   np.average(grp['TO_pct'],  weights=wts),
        })
    return pd.DataFrame(agg)

print("Computing stats...")
m_stats = compute_team_stats(m_reg_detailed)
w_stats = compute_team_stats(w_reg_detailed)

# ── 4. STRENGTH OF SCHEDULE ───────────────────────────────────
def compute_sos(reg_compact, team_stats):
    net_lookup = team_stats.set_index(['Season', 'TeamID'])['NetEff'].to_dict()
    rows = []
    for _, r in reg_compact.iterrows():
        s = r['Season']
        rows.append({'Season': s, 'TeamID': r['WTeamID'], 'OppNet': net_lookup.get((s, r['LTeamID']), 0.0)})
        rows.append({'Season': s, 'TeamID': r['LTeamID'], 'OppNet': net_lookup.get((s, r['WTeamID']), 0.0)})
    sos = pd.DataFrame(rows).groupby(['Season', 'TeamID'])['OppNet'].mean().reset_index()
    sos.rename(columns={'OppNet': 'SoS'}, inplace=True)
    return sos

m_stats = m_stats.merge(compute_sos(m_reg_compact, m_stats), on=['Season', 'TeamID'], how='left')
w_stats = w_stats.merge(compute_sos(w_reg_compact, w_stats), on=['Season', 'TeamID'], how='left')

# ── 5. MASSEY ORDINALS ───────────────────────────────────────
def get_massey(massey_df):
    df = massey_df[massey_df['RankingDayNum'] <= 133].copy()
    pref = [s for s in ['POM','SAG','MOR','DOL','RPI'] if s in df['SystemName'].unique()]
    if not pref:
        pref = list(df['SystemName'].unique()[:5])
    df = df[df['SystemName'].isin(pref)]
    last_day = df.groupby(['Season', 'SystemName'])['RankingDayNum'].transform('max')
    df = df[df['RankingDayNum'] == last_day]
    comp = df.groupby(['Season', 'TeamID'])['OrdinalRank'].mean().reset_index()
    comp.rename(columns={'OrdinalRank': 'MasseyRank'}, inplace=True)
    return comp

m_stats = m_stats.merge(get_massey(m_massey_raw), on=['Season', 'TeamID'], how='left')
# Women's: derive from NetEff rank
w_stats['MasseyRank'] = w_stats.groupby('Season')['NetEff'].rank(ascending=False)

# ── 6. ELO RATINGS ───────────────────────────────────────────
def compute_elo(reg_compact, K=20, revert=0.33):
    DEFAULT = 1500.0
    elo = {}
    elo_history = {}
    for season, sdf in reg_compact.sort_values(['Season','DayNum']).groupby('Season'):
        for tid in elo:
            elo[tid] = elo[tid] * (1 - revert) + DEFAULT * revert
        for _, row in sdf.iterrows():
            wt, lt = row['WTeamID'], row['LTeamID']
            ew = elo.get(wt, DEFAULT); el = elo.get(lt, DEFAULT)
            exp_w = 1 / (1 + 10 ** ((el - ew) / 400))
            margin = row['WScore'] - row['LScore']
            mov = np.log(abs(margin) + 1) * (2.2 / (abs(ew - el) * 0.001 + 2.2))
            k = K * mov
            elo[wt] = ew + k * (1 - exp_w)
            elo[lt] = el + k * (0 - (1 - exp_w))
        for tid, e in elo.items():
            elo_history[(season, tid)] = e
    rows = [{'Season': s, 'TeamID': t, 'Elo': e} for (s, t), e in elo_history.items()]
    return pd.DataFrame(rows)

print("Computing Elo...")
m_elo = compute_elo(m_reg_compact)
w_elo = compute_elo(w_reg_compact)
m_stats = m_stats.merge(m_elo, on=['Season', 'TeamID'], how='left')
w_stats = w_stats.merge(w_elo, on=['Season', 'TeamID'], how='left')

# ── 7. BUILD MATCHUP DATASET ─────────────────────────────────
STAT_COLS = ['Elo', 'OffEff', 'DefEff', 'NetEff', 'Pace', 'TS_pct', 'Reb_pct', 'TO_pct', 'SoS', 'MasseyRank']
LOWER_BETTER = {'DefEff', 'TO_pct', 'MasseyRank'}

def build_matchup_features(tourney_df, stats_df, seeds_df, gender_val):
    df = tourney_df[tourney_df['Season'] >= 2003].copy()
    # Use dict for O(1) lookup — THIS is the fix
    stats_dict = {}
    for _, r in stats_df.iterrows():
        stats_dict[(r['Season'], r['TeamID'])] = r

    seed_dict = {}
    for _, r in seeds_df.iterrows():
        seed_dict[(r['Season'], r['TeamID'])] = r['SeedNum']

    records = []
    for _, row in df.iterrows():
        season = row['Season']
        wt, lt = row['WTeamID'], row['LTeamID']
        t1, t2 = (wt, lt) if wt < lt else (lt, wt)
        target = 1 if wt < lt else 0

        s1 = stats_dict.get((season, t1))
        s2 = stats_dict.get((season, t2))
        if s1 is None or s2 is None:
            continue

        feat = {'Season': season, 'T1': t1, 'T2': t2, 'Target': target, 'Gender': gender_val}
        for col in STAT_COLS:
            v1 = s1.get(col, np.nan) if isinstance(s1, dict) else s1[col] if col in s1.index else np.nan
            v2 = s2.get(col, np.nan) if isinstance(s2, dict) else s2[col] if col in s2.index else np.nan
            feat[f'{col}_diff'] = (v2 - v1) if col in LOWER_BETTER else (v1 - v2)

        seed1 = seed_dict.get((season, t1), 8)
        seed2 = seed_dict.get((season, t2), 8)
        feat['Seed1'] = seed1; feat['Seed2'] = seed2
        feat['Seed_diff'] = seed2 - seed1
        feat['AbsSeed_diff'] = abs(seed1 - seed2)

        elo1_val = s1['Elo'] if 'Elo' in s1.index and not pd.isna(s1['Elo']) else 1500.0
        elo2_val = s2['Elo'] if 'Elo' in s2.index and not pd.isna(s2['Elo']) else 1500.0
        feat['Elo1'] = elo1_val; feat['Elo2'] = elo2_val
        records.append(feat)

    result = pd.DataFrame(records)
    print(f"[{'M' if gender_val==0 else 'W'}] Matchup rows: {len(result)}, lower-ID win rate: {result['Target'].mean():.3f}")
    return result

m_matchups = build_matchup_features(m_tourney, m_stats, m_seeds_clean, 0)
w_matchups = build_matchup_features(w_tourney, w_stats, w_seeds_clean, 1)
all_matchups = pd.concat([m_matchups, w_matchups], ignore_index=True)
print(f"Total matchups: {len(all_matchups)}")

# ── 8. FEATURES & LABELS ─────────────────────────────────────
FEATURE_COLS = [
    'Elo_diff', 'OffEff_diff', 'DefEff_diff', 'NetEff_diff', 'Pace_diff',
    'TS_pct_diff', 'Reb_pct_diff', 'TO_pct_diff', 'SoS_diff', 'MasseyRank_diff',
    'Seed_diff', 'AbsSeed_diff', 'Gender'
]
X = all_matchups[FEATURE_COLS].fillna(0).values
y = all_matchups['Target'].values
print(f"X shape: {X.shape}, win rate: {y.mean():.3f}")

# ── 9. LIGHTGBM CROSS-VALIDATION ─────────────────────────────
lgb_params = dict(
    objective='binary', metric='binary_logloss', learning_rate=0.02,
    n_estimators=1000, num_leaves=31, min_child_samples=10,
    subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.5,
    random_state=SEED, verbose=-1, n_jobs=-1
)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof_lgb = np.zeros(len(X))
models_lgb = []

for fold, (tr, val) in enumerate(skf.split(X, y)):
    m = LGBMClassifier(**lgb_params)
    m.fit(X[tr], y[tr], eval_set=[(X[val], y[val])],
          callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(False)])
    oof_lgb[val] = m.predict_proba(X[val])[:, 1]
    models_lgb.append(m)
    print(f"Fold {fold+1} | iter={m.best_iteration_} | Brier={brier_score_loss(y[val], oof_lgb[val]):.4f}")

print(f"OOF LightGBM Brier: {brier_score_loss(y, oof_lgb):.4f}")

# ── 10. ELO LOGISTIC MODEL ────────────────────────────────────
elo_diff_train = all_matchups['Elo1'].fillna(1500).values - all_matchups['Elo2'].fillna(1500).values
best_scale = minimize_scalar(
    lambda sc: brier_score_loss(y, expit(elo_diff_train / sc)),
    bounds=(50, 1000), method='bounded'
).x
oof_elo = expit(elo_diff_train / best_scale)
print(f"Elo scale={best_scale:.1f}, Brier={brier_score_loss(y, oof_elo):.4f}")

# ── 11. CALIBRATION ───────────────────────────────────────────
ir_lgb = IsotonicRegression(out_of_bounds='clip').fit(oof_lgb, y)
ir_elo = IsotonicRegression(out_of_bounds='clip').fit(oof_elo, y)
cal_lgb = ir_lgb.predict(oof_lgb)
cal_elo = ir_elo.predict(oof_elo)

# Auto-tune ensemble weight
best_bs, best_w = 999, 0.7
for w in np.arange(0.0, 1.05, 0.05):
    bs = brier_score_loss(y, w * cal_lgb + (1 - w) * cal_elo)
    if bs < best_bs:
        best_bs, best_w = bs, w
W_LGB, W_ELO = best_w, 1 - best_w
print(f"Best ensemble: LGB={W_LGB:.2f}, Elo={W_ELO:.2f}, Brier={best_bs:.4f}")

# ── 12. BUILD SUBMISSION FEATURES ────────────────────────────
parts = sample_sub['ID'].str.split('_', expand=True)
sub_df = sample_sub.copy()
sub_df['Season'] = parts[0].astype(int)
sub_df['T1']     = parts[1].astype(int)
sub_df['T2']     = parts[2].astype(int)
sub_df['Gender'] = ((sub_df['T1'] >= 3000) | (sub_df['T2'] >= 3000)).astype(int)

PRED_SEASON = sub_df['Season'].iloc[0]

# Build stat dicts for prediction season
def make_stat_dict(stats_df, season):
    return {r['TeamID']: r for _, r in stats_df[stats_df['Season'] == season].iterrows()}

def make_seed_dict(seeds_df, season):
    return seeds_df[seeds_df['Season'] == season].set_index('TeamID')['SeedNum'].to_dict()

m_stat_pred = make_stat_dict(m_stats, PRED_SEASON)
w_stat_pred = make_stat_dict(w_stats, PRED_SEASON)
m_seed_pred = make_seed_dict(m_seeds_clean, PRED_SEASON)
w_seed_pred = make_seed_dict(w_seeds_clean, PRED_SEASON)

def safe_get(stat_row, col):
    if stat_row is None:
        return np.nan
    try:
        v = stat_row[col]
        return np.nan if pd.isna(v) else float(v)
    except:
        return np.nan

sub_records = []
for _, row in sub_df.iterrows():
    t1, t2, gender = row['T1'], row['T2'], row['Gender']
    sd = w_stat_pred if gender == 1 else m_stat_pred
    seed_d = w_seed_pred if gender == 1 else m_seed_pred
    s1, s2 = sd.get(t1), sd.get(t2)

    feat = {'ID': row['ID'], 'Gender': gender}
    for col in STAT_COLS:
        v1, v2 = safe_get(s1, col), safe_get(s2, col)
        feat[f'{col}_diff'] = (v2 - v1) if col in LOWER_BETTER else (v1 - v2)

    seed1, seed2 = seed_d.get(t1, 8), seed_d.get(t2, 8)
    feat['Seed1'] = seed1; feat['Seed2'] = seed2
    feat['Seed_diff'] = seed2 - seed1
    feat['AbsSeed_diff'] = abs(seed1 - seed2)
    feat['Elo1'] = safe_get(s1, 'Elo') or 1500.0
    feat['Elo2'] = safe_get(s2, 'Elo') or 1500.0
    sub_records.append(feat)

sub_feat = pd.DataFrame(sub_records)

# ── 13. PREDICT & SUBMIT ─────────────────────────────────────
X_sub = sub_feat[FEATURE_COLS].fillna(0).values

lgb_raw = np.mean([m.predict_proba(X_sub)[:, 1] for m in models_lgb], axis=0)

elo1 = np.where(sub_feat['Elo1'].isna(), 1500.0, sub_feat['Elo1'].values)
elo2 = np.where(sub_feat['Elo2'].isna(), 1500.0, sub_feat['Elo2'].values)
elo_raw = expit((elo1 - elo2) / best_scale)

# Clamp both raw arrays to the training range of the isotonic regressors
lgb_raw  = np.clip(lgb_raw,  0.001, 0.999)
elo_raw  = np.clip(elo_raw,  0.001, 0.999)

# Replace any residual NaN/inf with 0.5 before calibrating
lgb_raw  = np.where(np.isfinite(lgb_raw),  lgb_raw,  0.5)
elo_raw  = np.where(np.isfinite(elo_raw),  elo_raw,  0.5)

lgb_cal  = ir_lgb.predict(lgb_raw)
elo_cal  = ir_elo.predict(elo_raw)

final_preds = np.clip(W_LGB * lgb_cal + W_ELO * elo_cal, 0.05, 0.95)

submission = pd.DataFrame({'ID': sub_feat['ID'], 'Pred': final_preds})
submission.to_csv('/kaggle/working/submission.csv', index=False)

print(f"\n✅ Submission saved! Rows={len(submission)}")
print(f"   Pred range: [{final_preds.min():.3f}, {final_preds.max():.3f}], mean={final_preds.mean():.3f}")
print(submission.head(5).to_string(index=False))

Loaded. Sub rows: 519144
Computing stats...
Computing Elo...
[M] Matchup rows: 1449, lower-ID win rate: 0.500
[W] Matchup rows: 961, lower-ID win rate: 0.505
Total matchups: 2410
X shape: (2410, 13), win rate: 0.502
Fold 1 | iter=155 | Brier=0.1732
Fold 2 | iter=101 | Brier=0.1772
Fold 3 | iter=134 | Brier=0.1767
Fold 4 | iter=131 | Brier=0.1711
Fold 5 | iter=195 | Brier=0.1708
OOF LightGBM Brier: 0.1738
Elo scale=122.4, Brier=0.1753
Best ensemble: LGB=0.55, Elo=0.45, Brier=0.1682

✅ Submission saved! Rows=519144
   Pred range: [0.050, 0.950], mean=0.479
            ID     Pred
2022_1101_1102 0.888109
2022_1101_1103 0.453042
2022_1101_1104 0.352605
2022_1101_1105 0.872043
2022_1101_1106 0.889286
